# Figure2_rank2_to_rank10_program_landscapes_and_rank8_drivers

In [1]:

from pathlib import Path
import os, re, warnings, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from scipy.stats import hypergeom
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import roc_curve, auc

try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    from torch.utils.data import DataLoader, TensorDataset, random_split
    HAS_TORCH = True
except Exception as e:
    HAS_TORCH = False
    print('Torch unavailable:', e)

try:
    from tensorly.decomposition import parafac
    from tensorly.cp_tensor import cp_to_tensor
    HAS_TENSORLY = True
except Exception as e:
    HAS_TENSORLY = False
    print('Tensorly unavailable:', e)

project_dir = Path('/Users/sidaye/Documents/python/ST_MultiCAST')
input_dir = project_dir / 'Input'
base_output_dir = project_dir / 'Output'
model_comparison_dir = base_output_dir / 'model comparison'
ai_output_dir = base_output_dir / 'AI_spatiotemporal_models_python'
Spacepoints = ['st','SI1','SI2','SI3','SI4','SI5','SI6','SI7','SI8','SI9','ce','co']
Full_Timepoints = ['1h','3h','6h','12h','24h']
feature_order = [f'{t}_{s}' for t in Full_Timepoints for s in Spacepoints]

plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
plt.rcParams['font.family'] = 'DejaVu Sans'

def save_pdf(fig, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=300, bbox_inches='tight', transparent=True)
    plt.close(fig)

def build_data():
    df = pd.read_csv(input_dir / 'Spatial_temporal_MultiSCAST_FC_final_capping.csv')
    df['Gene'] = df['Gene'].astype(str)
    df['Time'] = df['Time'].astype(str)
    df['Space'] = df['Space'].astype(str)
    df = df[df['Time'].isin(Full_Timepoints) & df['Space'].isin(Spacepoints)].copy()
    df['Feature'] = df['Time'] + '_' + df['Space']
    df = df.groupby(['Gene','Time','Space','Feature'], as_index=False).agg(logFC=('logFC','mean'))
    wide = df.pivot_table(index='Gene', columns='Feature', values='logFC', aggfunc='mean')
    wide = wide[[f for f in feature_order if f in wide.columns]].dropna(axis=0, how='any')
    X_raw_df = wide.copy()
    X_raw = X_raw_df.values.astype(float)
    row_mean = X_raw.mean(axis=1, keepdims=True)
    row_std = X_raw.std(axis=1, keepdims=True)
    row_std[row_std == 0] = 1.0
    X_scaled = np.nan_to_num((X_raw - row_mean) / row_std)
    X_scaled_df = pd.DataFrame(X_scaled, index=X_raw_df.index.astype(str), columns=X_raw_df.columns)
    return X_raw_df, X_scaled_df, row_mean, row_std

def vector_to_landscape(vector, columns=None):
    if columns is None:
        columns = feature_order
    s = pd.Series(np.asarray(vector, dtype=float), index=columns)
    return s.reindex(feature_order).values.reshape(len(Full_Timepoints), len(Spacepoints))

def load_category_table():
    cat = pd.read_excel(input_dir / 'putative_driver_gene_categories_12class.xlsx', sheet_name=0)
    cat['locus_ID'] = cat['locus_ID'].astype(str)
    col = 'Putative_driver_category'
    cat_map = cat.set_index('locus_ID')[col].dropna().to_dict()
    cats = sorted(pd.Series(cat_map).dropna().unique().tolist())
    palette = plt.cm.tab20(np.linspace(0, 1, max(20, len(cats))))
    color_map = {c: palette[i] for i, c in enumerate(cats)}
    default = '#2f6db3'
    return cat_map, color_map, default

def load_annotation():
    ann = pd.read_csv(input_dir / 'new_annotations_with_uniprot_names.csv')
    ann['locus_ID'] = ann['locus_ID'].astype(str)
    display_cols = ['gene_name','uniprot_gene_name','gene_name_old','KEGG_VC_number']
    def display(row):
        for c in display_cols:
            v = row.get(c, np.nan)
            if pd.notna(v) and str(v).strip() and str(v).lower() != 'nan':
                return str(v)
        return str(row['locus_ID'])
    ann['Gene_display'] = ann.apply(display, axis=1)
    text_cols = [c for c in ann.columns if c != 'locus_ID']
    ann['Annotation_text'] = ann[text_cols].astype(str).replace('nan','', regex=False).agg(' | '.join, axis=1)
    return ann

def phase_for_time(t):
    return {'1h':'Early','3h':'Early','6h':'Middle','12h':'Middle','24h':'Late'}.get(t, '')

def niche_for_space(s):
    if s == 'st': return 'stomach'
    if str(s).startswith('SI'): return 'small_intestine'
    if s == 'ce': return 'cecum'
    if s == 'co': return 'colon'
    return s

outdir = base_output_dir / 'program_landscapes2'
outdir.mkdir(parents=True, exist_ok=True)
land = pd.read_csv(model_comparison_dir / 'PCA_CP_VAE_CPVAE_best_CPVAE_rank_program_landscapes_long.csv')
model_order = ['PCA','CP','VAE','weighted_CPVAE']
model_titles = {'PCA':'PCA','CP':'CP','VAE':'VAE','weighted_CPVAE':'weighted CPVAE'}
for rank in range(2, 11):
    rows=[]
    recs=[]
    for idx in range(1, rank+1):
        row=[]
        for model in model_order:
            sub = land[(land['Model'].eq(model)) & (land['Program'].str.extract(r'(\d+)')[0].astype(int).eq(idx))]
            if sub.empty:
                row.append(None); continue
            mat = sub.pivot(index='Time', columns='Space', values='Value').reindex(index=Full_Timepoints, columns=Spacepoints).values.astype(float)
            row.append((f"{model_titles[model]} {idx}", mat))
            tmp=sub.copy(); tmp['Requested_rank']=rank; recs.append(tmp)
        rows.append(row)
    vals=np.concatenate([((m-np.nanmedian(m))/(np.nanstd(m) or 1)).ravel() for row in rows for item in row if item is not None for _,m in [item]])
    vmax=min(max(np.nanquantile(np.abs(vals),0.98),1.5),3.0)
    fig, axes = plt.subplots(rank, 4, figsize=(14.2, 2.05*rank), squeeze=False)
    im=None
    for r,row in enumerate(rows):
        for c,item in enumerate(row):
            ax=axes[r,c]
            if item is None:
                ax.axis('off'); continue
            title, mat=item
            z=(mat-np.nanmedian(mat))/(np.nanstd(mat) or 1)
            im=ax.imshow(z, aspect='auto', cmap='coolwarm', vmin=-vmax, vmax=vmax)
            ax.set_title(title, fontsize=8.5, pad=6)
            ax.set_xticks(range(len(Spacepoints))); ax.set_xticklabels(Spacepoints, rotation=45, ha='right', fontsize=6)
            ax.set_yticks(range(len(Full_Timepoints))); ax.set_yticklabels(Full_Timepoints, fontsize=6)
    fig.suptitle(f'Rank-{rank} program landscapes across PCA, CP, VAE, and weighted CPVAE', fontsize=12.5, y=0.985)
    fig.subplots_adjust(top=0.92, bottom=0.06, left=0.06, right=0.88, hspace=0.72, wspace=0.30)
    cax=fig.add_axes([0.90,0.18,0.014,0.64]); fig.colorbar(im,cax=cax,label='Per-panel z-score of program loading')
    save_pdf(fig, outdir / f'Figure2_rank{rank}_PCA_CP_VAE_CPVAE_program_landscapes.pdf')
    if recs:
        pd.concat(recs).to_csv(outdir / f'Figure2_rank{rank}_program_landscapes_long.csv', index=False)

# Rank8 model fitting for driver genes
X_raw_df, X_scaled_df, row_mean, row_std = build_data()
gene_ids = X_scaled_df.index.to_list(); X_scaled=X_scaled_df.values
ann = load_annotation(); ann_display = ann.set_index('locus_ID')['Gene_display'].to_dict()
cat_map, cat_colors, default_color = load_category_table()
rank=8
# PCA scores
pca=PCA(n_components=rank).fit(X_scaled)
pca_scores=pca.transform(X_scaled)
# CP factors
if not HAS_TENSORLY: raise RuntimeError('tensorly required')
tensor=np.zeros((len(gene_ids), len(Full_Timepoints), len(Spacepoints)))
for ti,t in enumerate(Full_Timepoints):
    for si,s in enumerate(Spacepoints):
        tensor[:,ti,si]=X_scaled_df[f'{t}_{s}'].values
cp=parafac(tensor, rank=rank, n_iter_max=120, tol=1e-5, init='svd', random_state=1, normalize_factors=False)
weights, factors=cp; cp_gene=factors[0]*weights.reshape(1,-1)
# VAE/CPVAE lightweight gene scores: use existing CPVAE rank6 if no torch? else train compact autoencoder style VAE
if not HAS_TORCH:
    vae_scores=np.zeros((len(gene_ids), rank)); cpvae_scores=np.zeros((len(gene_ids), rank))
else:
    torch.manual_seed(9); np.random.seed(9)
    device=torch.device('cpu')
    data=torch.tensor(X_scaled, dtype=torch.float32)
    ds=TensorDataset(data); loader=DataLoader(ds, batch_size=128, shuffle=True)
    class VAE(nn.Module):
        def __init__(self, inp, latent):
            super().__init__(); self.latent_dim=latent
            self.enc=nn.Sequential(nn.Linear(inp,128), nn.ReLU(), nn.Linear(128,64), nn.ReLU())
            self.mu=nn.Linear(64,latent); self.lv=nn.Linear(64,latent)
            self.dec=nn.Sequential(nn.Linear(latent,64), nn.ReLU(), nn.Linear(64,128), nn.ReLU(), nn.Linear(128,inp))
        def encode(self,x): h=self.enc(x); return self.mu(h), self.lv(h)
        def forward(self,x):
            mu,lv=self.encode(x); z=mu + torch.randn_like(mu)*torch.exp(0.5*lv); return self.dec(z), mu, lv
    def train_vae():
        m=VAE(X_scaled.shape[1], rank); opt=optim.Adam(m.parameters(), lr=1e-3)
        for ep in range(35):
            for (xb,) in loader:
                opt.zero_grad(); rec,mu,lv=m(xb); loss=((rec-xb)**2).sum()+0.001*(-0.5*torch.sum(1+lv-mu.pow(2)-lv.exp())); loss.backward(); opt.step()
        with torch.no_grad(): mu,_=m.encode(data)
        return mu.numpy()
    vae_scores=train_vae()

    class CPStructuredVAE(nn.Module):
        def __init__(self, inp, latent, n_time, n_space):
            super().__init__(); self.latent_dim=latent; self.n_time=n_time; self.n_space=n_space
            self.enc=nn.Sequential(nn.Linear(inp,128), nn.ReLU(), nn.Linear(128,64), nn.ReLU())
            self.mu=nn.Linear(64,latent); self.lv=nn.Linear(64,latent)
            self.time_factor=nn.Parameter(torch.randn(n_time, latent)*0.05)
            self.space_factor=nn.Parameter(torch.randn(n_space, latent)*0.05)
            self.feature_bias=nn.Parameter(torch.zeros(n_time*n_space))
            self.residual=nn.Sequential(nn.Linear(latent,64), nn.ReLU(), nn.Linear(64,inp))
            self.residual_scale=nn.Parameter(torch.tensor(0.25))
        def encode(self,x):
            h=self.enc(x); return self.mu(h), self.lv(h)
        def cp_decode(self,z):
            basis=torch.einsum('tr,sr->rts', self.time_factor, self.space_factor).reshape(self.latent_dim, self.n_time*self.n_space)
            return z @ basis + self.feature_bias
        def decode(self,z,include_residual=True):
            cp=self.cp_decode(z)
            if not include_residual: return cp
            return cp + torch.tanh(self.residual_scale)*self.residual(z)
        def forward(self,x):
            mu,lv=self.encode(x); z=mu + torch.randn_like(mu)*torch.exp(0.5*lv); return self.decode(z), mu, lv
    def offdiag_ms(mat):
        gram=mat.T@mat; off=gram-torch.diag(torch.diag(gram)); return torch.mean(off.pow(2))
    def train_cpvae_rank8():
        torch.manual_seed(18); np.random.seed(18)
        m=CPStructuredVAE(X_scaled.shape[1], rank, len(Full_Timepoints), len(Spacepoints))
        with torch.no_grad():
            m.time_factor.copy_(torch.tensor(factors[1], dtype=torch.float32))
            m.space_factor.copy_(torch.tensor(factors[2], dtype=torch.float32))
        opt=optim.Adam(m.parameters(), lr=1e-3)
        for ep in range(70):
            for (xb,) in loader:
                opt.zero_grad(); rec,mu,lv=m(xb)
                total=((rec-xb)**2).sum()
                cp_only=m.decode(mu, include_residual=False)
                cp_loss=((cp_only-xb)**2).sum()
                kl=-0.5*torch.sum(1+lv-mu.pow(2)-lv.exp())
                tnorm=nn.functional.normalize(m.time_factor, dim=0); snorm=nn.functional.normalize(m.space_factor, dim=0)
                orth=offdiag_ms(tnorm)+offdiag_ms(snorm)
                residual_pen=torch.mean(torch.abs(torch.tanh(m.residual_scale)*m.residual(mu)))
                loss=total+0.35*cp_loss+0.001*kl+0.01*orth*xb.shape[0]+0.02*residual_pen*xb.shape[0]
                loss.backward(); opt.step()
        with torch.no_grad():
            mu,_=m.encode(data)
        return mu.numpy()
    cpvae_scores=train_cpvae_rank8()

# Save true rank8 CPVAE encoder latent means directly from the CPVAE model.
# These columns are encoder mu dimensions, not decoded program landscapes.
cpvae_latent_df = pd.DataFrame(cpvae_scores, columns=[f'CPVAE{i+1}' for i in range(rank)])
cpvae_latent_df['Gene'] = gene_ids
cpvae_latent_df['Gene_display'] = [ann_display.get(gene, gene) for gene in gene_ids]
cpvae_latent_df = cpvae_latent_df[['Gene','Gene_display'] + [f'CPVAE{i+1}' for i in range(rank)]]
cpvae_latent_df.to_csv(outdir / 'Figure2_rank8_CPVAE_encoder_latent_mean.csv', index=False)

score_tables={'PCA':pca_scores,'CP':cp_gene,'VAE':vae_scores,'CPVAE':cpvae_scores}
score_records=[]
for model, scores in score_tables.items():
    for i,gene in enumerate(gene_ids):
        rec={'Model':model,'Gene':gene,'Gene_display':ann_display.get(gene,gene)}
        for k in range(rank): rec[f'Program{k+1}']=scores[i,k]
        score_records.append(rec)
pd.DataFrame(score_records).to_csv(outdir / 'Figure2_rank8_gene_program_scores_four_models.csv', index=False)
records=[]
for model, scores in score_tables.items():
    for k in range(rank):
        for direction, order in [('Positive',False),('Negative',True)]:
            idx=np.argsort(scores[:,k])[:10] if direction=='Negative' else np.argsort(scores[:,k])[-10:][::-1]
            for i in idx:
                gene=gene_ids[i]
                cat=cat_map.get(gene, 'Uncategorized')
                records.append({'Model':model,'Program':f'{model}{k+1}','Component':k+1,'Direction':direction,'Gene':gene,'Gene_display':ann_display.get(gene,gene),'Loading':scores[i,k],'Abs_loading':abs(scores[i,k]),'Driver_category':cat})
drivers=pd.DataFrame(records)
drivers.to_csv(outdir / 'Figure2_rank8_top10_driver_genes_four_models_with_12class_categories.csv', index=False)
# driver plot
cats=['Uncategorized']+sorted([c for c in drivers['Driver_category'].dropna().unique() if c!='Uncategorized'])
colors={c: cat_colors.get(c, default_color) for c in cats}; colors['Uncategorized']=default_color
for model in ['PCA','CP','VAE','CPVAE']:
    sub=drivers[drivers['Model'].eq(model)].copy()
    fig, axes=plt.subplots(rank,2,figsize=(16.4,2.15*rank),squeeze=False)
    for k in range(1,rank+1):
        for c,direction in enumerate(['Positive','Negative']):
            ax=axes[k-1,c]
            ss=sub[(sub['Component'].eq(k)) & (sub['Direction'].eq(direction))].copy()
            ss=ss.sort_values('Loading', ascending=(direction=='Positive'))
            labels=ss['Gene_display'].fillna(ss['Gene']).astype(str)
            y=np.arange(len(ss)); ax.barh(y, ss['Loading'], color=[colors.get(x,default_color) for x in ss['Driver_category']], alpha=0.9)
            ax.axvline(0,color='black',lw=0.7); ax.set_yticks(y); ax.set_yticklabels(labels,fontsize=6.0); ax.tick_params(axis='y', pad=2)
            ax.set_title(f'{model} component {k} top {direction.lower()} genes',fontsize=8.5,pad=6)
    handles=[Patch(facecolor=colors[c],label=c) for c in cats]
    fig.legend(handles=handles,frameon=False,fontsize=6.5,loc='lower center',ncol=3,bbox_to_anchor=(0.5,0.005))
    fig.suptitle(f'Rank8 {model} top10 driver genes colored by 12-class category',fontsize=12,y=0.995)
    fig.subplots_adjust(top=0.945,bottom=0.11,left=0.16,right=0.97,hspace=0.66,wspace=0.68)
    save_pdf(fig, outdir / f'Figure2_rank8_{model}_top10_driver_genes_12class.pdf')
print(outdir)


/Users/sidaye/Documents/python/ST_MultiCAST/Output/program_landscapes2
